# Where could c14_age_bp, c14_error, d13C, pMC_value, pMC_error, cal_68 and cal_95 live in SEAD_staging?

This notebook explores the SEAD_staging schema for possible homes for these Strucke columns:
- dedicated columns created specifically for one of these values
- generic "measurement/value" tables where such a value could be stored via a foreign key (e.g. tied to a method, entity or dataset), rather than a purpose-built column


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Connect to sead_staging database


In [2]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## List all tables in the public schema


In [3]:
tables = pd.read_sql(
    "select table_name from information_schema.tables where table_schema = 'public' order by table_name",
    engine,
)
print(f'{len(tables)} tables/views in the public schema')
tables


181 tables/views in the public schema


,table_name
0,master_set_reference
1,taxon_view
2,tbl_abundance_elements
3,tbl_abundance_ident_levels
4,tbl_abundance_modifications
...,...
176,view_typed_analysis_tables
177,view_typed_analysis_values
178,view_with_abundances
179,view_with_references


## Search for columns whose name hints at these values

Keywords: age, error, d13c, delta, pmc, cal, c14, carbon, isotope, uncertain, value.


In [4]:
candidate_columns = pd.read_sql(
    text(
        """
        select table_name, column_name, data_type
        from information_schema.columns
        where table_schema = 'public'
          and (
            column_name ilike '%age%' or
            column_name ilike '%error%' or
            column_name ilike '%d13c%' or
            column_name ilike '%delta%' or
            column_name ilike '%pmc%' or
            column_name ilike '%cal%' or
            column_name ilike '%c14%' or
            column_name ilike '%carbon%' or
            column_name ilike '%isotope%' or
            column_name ilike '%uncertain%' or
            column_name ilike '%value%'
          )
        order by table_name, ordinal_position
        """
    ),
    engine,
)
print(f"{len(candidate_columns)} candidate columns across {candidate_columns['table_name'].nunique()} tables")
candidate_columns


176 candidate columns across 69 tables


,table_name,column_name,data_type
0,tbl_abundance_properties,property_value,text
1,tbl_age_types,age_type_id,integer
2,tbl_age_types,age_type,character varying
3,tbl_aggregate_sample_ages,aggregate_sample_age_id,integer
4,tbl_aggregate_sample_ages,analysis_entity_age_id,integer
...,...,...,...
171,view_with_abundances,locality,text
172,view_with_abundances,language,text
173,view_with_times,age_older,numeric
174,view_with_times,age_younger,numeric


### `tbl_geochronology` looks like the dedicated home for c14_age_bp, c14_error and d13C

It has `age`, `error_older`/`error_younger` and `delta_13c` columns, plus `dating_lab_id` and `lab_number` (matching Strucke's `lab_no`) and `analysis_entity_id` linking it to a physical sample.

No column anywhere in the schema is named for pMC or for a calibrated 68%/95% range, though — worth keeping in mind while reading the rest of this notebook.


In [5]:
geochronology_schema = pd.read_sql(
    "select column_name, data_type from information_schema.columns where table_name = 'tbl_geochronology' order by ordinal_position",
    engine,
)
geochronology_schema


,column_name,data_type
0,geochron_id,integer
1,analysis_entity_id,bigint
2,dating_lab_id,integer
3,lab_number,character varying
4,age,numeric
5,error_older,numeric
6,error_younger,numeric
7,delta_13c,numeric
8,notes,text
9,date_updated,timestamp with time zone


In [6]:
geochronology = pd.read_sql('select * from public.tbl_geochronology', engine)
print(f'{len(geochronology)} rows in tbl_geochronology')
print(f"{geochronology['delta_13c'].notna().sum()} rows have delta_13c populated")
geochronology[['geochron_id', 'dating_lab_id', 'lab_number', 'age', 'error_older', 'error_younger', 'delta_13c']].sample(10, random_state=1)


1463 rows in tbl_geochronology
0 rows have delta_13c populated


,geochron_id,dating_lab_id,lab_number,age,error_older,error_younger,delta_13c
719,720,774,K-4819,3780.0,85.0,85.0,None
683,684,883,UB-4581,7894.0,35.0,35.0,None
503,504,863,SRR-3461,11060.0,70.0,70.0,None
424,424,916,HUTH-3212,167500.0,5400.0,3500.0,None
846,845,743,GrN-18157,26430.0,240.0,240.0,None
860,858,743,GrN-18149,24590.0,120.0,120.0,None
1078,1080,702,Birm-409,42000.0,1000.0,1000.0,None
1080,1082,702,Birm-409,42000.0,1000.0,1000.0,None
1132,1134,700,Beta-316485,6250.0,40.0,40.0,None
895,893,687,AAR-1279,6900.0,100.0,100.0,None


## The generic analysis_entity → analysis_value pattern

SEAD also has a generic EAV-style chain for measurements that don't get a purpose-built column:

`tbl_datasets` → `tbl_dataset_methods` (which `tbl_methods` was used) → `tbl_analysis_entities` (one per physical_sample+dataset) → `tbl_analysis_values` (one row per measured property, tagged with a `value_class_id`) → a typed subtype table holding the actual value: `tbl_analysis_numerical_values`, `tbl_analysis_integer_values`, `tbl_analysis_categorical_values`, `tbl_analysis_boolean_values`, `tbl_analysis_dating_ranges`, `tbl_analysis_numerical_ranges`, `tbl_analysis_integer_ranges`, `tbl_analysis_identifiers` or `tbl_analysis_notes`.

If any of our target columns are stored generically rather than in a dedicated column, this is the mechanism that would hold them, tied together via foreign keys rather than a named column.


In [7]:
c14_methods = pd.read_sql(
    text(
        """
        select method_id, method_name, method_abbrev_or_alt_name, method_group_id
        from tbl_methods
        where method_name ilike '%radiocarbon%'
           or method_name ilike '%c14%'
           or method_name ilike '%carbon%'
           or method_name ilike '%calib%'
        order by method_id
        """
    ),
    engine,
)
c14_methods


,method_id,method_name,method_abbrev_or_alt_name,method_group_id
0,38,C14 Accelerator dating,C14 AMS,3
1,39,C14 Conventional,C14,3
2,129,Archaeological period C14 years,ArchPerC14,19
3,132,Geological C14 period,GeolPerC14,19
4,136,Tephrochronology C14,TephraC14,20
5,148,Radiocarbon (14C Unspecified),C14 Unspec.,3
6,151,C14 Conventional,C14 Std,3
7,153,C14 dating of humic substances in sediment,C14 Humous,3
8,156,Calibrated radiocarbon date (method unspecified),Cal,20
9,157,Calibrated AMS radiocarbon date,CalAMS,20


### Are any of these methods actually used by a dataset?

If a method is never referenced in `tbl_dataset_methods`, the generic value tables can't currently hold anything tagged with it.


In [8]:
method_ids = ','.join(str(m) for m in c14_methods['method_id'])
method_usage = pd.read_sql(
    f"""
    select method_id, count(*) as dataset_count
    from tbl_dataset_methods
    where method_id in ({method_ids})
    group by method_id
    """,
    engine,
)
print(f"{len(method_usage)} of the {len(c14_methods)} C14-related methods are referenced in tbl_dataset_methods")
method_usage


0 of the 11 C14-related methods are referenced in tbl_dataset_methods


,method_id,dataset_count


### Row counts across the generic value-storage chain


In [9]:
generic_value_tables = pd.read_sql(
    """
    select 'tbl_dataset_methods' as table_name, count(*) as row_count from tbl_dataset_methods
    union all select 'tbl_analysis_entities', count(*) from tbl_analysis_entities
    union all select 'tbl_analysis_values', count(*) from tbl_analysis_values
    union all select 'tbl_analysis_numerical_values', count(*) from tbl_analysis_numerical_values
    union all select 'tbl_analysis_integer_values', count(*) from tbl_analysis_integer_values
    union all select 'tbl_analysis_dating_ranges', count(*) from tbl_analysis_dating_ranges
    union all select 'tbl_analysis_numerical_ranges', count(*) from tbl_analysis_numerical_ranges
    union all select 'tbl_analysis_integer_ranges', count(*) from tbl_analysis_integer_ranges
    """,
    engine,
)
generic_value_tables


,table_name,row_count
0,tbl_analysis_numerical_values,60
1,tbl_analysis_numerical_ranges,0
2,tbl_analysis_integer_ranges,0
3,tbl_dataset_methods,0
4,tbl_analysis_dating_ranges,7775
5,tbl_analysis_integer_values,25790
6,tbl_analysis_values,72183
7,tbl_analysis_entities,163168


`tbl_dataset_methods` being empty means the generic pathway isn't currently wired up to any radiocarbon method in this staging copy of the database, even though `tbl_analysis_dating_ranges` and `tbl_analysis_values` already hold rows for other purposes. So today, `tbl_geochronology` is the only place that actually holds C14 age/error/d13C data — the generic tables are a structural possibility for the future, not a current data source to reconcile against.


## Isotope-specific generic tables (another possible home for d13C)

Independently of `tbl_geochronology.delta_13c`, SEAD has a generic isotope-measurement pattern: `tbl_isotopes` (one row per measurement) → `tbl_isotope_measurements` → `tbl_isotope_types` (a full periodic table of elements, so "carbon" is just one of many) and `tbl_isotope_standards`.


In [10]:
isotope_tables = pd.read_sql(
    """
    select 'tbl_isotopes' as table_name, count(*) as row_count from tbl_isotopes
    union all select 'tbl_isotope_measurements', count(*) from tbl_isotope_measurements
    union all select 'tbl_isotope_standards', count(*) from tbl_isotope_standards
    """,
    engine,
)
isotope_tables


,table_name,row_count
0,tbl_isotopes,0
1,tbl_isotope_measurements,0
2,tbl_isotope_standards,1


## Units that might hint at pMC or calibrated-year storage

If pMC values were stored anywhere, we would expect a matching unit definition.


In [11]:
units = pd.read_sql('select unit_id, unit_name, unit_abbrev from public.tbl_units order by unit_id', engine)
print(f'{len(units)} units defined')
units


15 units defined


,unit_id,unit_name,unit_abbrev
0,1,metres,m
1,2,kilograms,kg
2,3,litres,l
3,4,Decimal degrees,dd
4,5,Units,NaN
5,6,Astronomical unit,NaN
6,7,14C years,C14yrs
7,8,Years,yrs
8,9,Degrees Celcius,°C
9,10,millimetres,mm


## Summary

- **c14_age_bp → `tbl_geochronology.age`**, **c14_error → `error_older`/`error_younger`**, **d13C → `delta_13c`**. This table already holds 1,463 real radiocarbon dates (recognisable lab codes like Ua-, GrN-, OxA-, Birm-), and also carries `lab_number` and `dating_lab_id`, matching Strucke's `lab_no`. `delta_13c` is present as a column but rarely populated in the sample checked.
- **pMC_value / pMC_error → no home found.** No column, method, or unit anywhere in the schema references percent modern carbon.
- **cal_68 / cal_95 → no dedicated home found either.** `tbl_analysis_dating_ranges` and `tbl_analysis_numerical_ranges` are structurally capable of holding a low/high range, and `tbl_methods` even has explicit "Calibrated radiocarbon date" method entries (156, 157) — but `tbl_dataset_methods` currently has zero rows for any C14-related method, so nothing is actually wired up to store a calibrated range today.
- The **generic analysis_entity → analysis_value chain** and the **generic isotope-measurement chain** are both real, working mechanisms elsewhere in SEAD, so they remain the most likely place pMC/cal ranges would go **if** they get modeled in the future — just not populated for radiocarbon today.


# Designing the generic-chain mapping for pMC_value, pMC_error, cal_68/95, and median_cal_year

The summary above established that none of these five columns have a dedicated column anywhere
in the schema, and that the generic `analysis_entity -> analysis_value` chain is structurally
capable but not currently wired up for any C14 method. This section works out concretely what
adding that wiring would look like: which `tbl_value_classes` rows would need to be created,
which subtype table each routes to, and which `tbl_methods` each hangs off.


## How the chain actually routes a value

Every `tbl_analysis_values` row points at a `value_class_id` (`tbl_value_classes`). Each value
class in turn points at a `value_type_id` (`tbl_value_types`) and a `method_id` (`tbl_methods`).
The value type's `base_type` determines which subtype table actually holds the value:

| `base_type` | subtype table |
|---|---|
| `integer` | `tbl_analysis_integer_values` (or `tbl_analysis_integer_ranges` for a low/high pair) |
| `decimal` | `tbl_analysis_numerical_values` (or `tbl_analysis_numerical_ranges`) |
| `boolean` | `tbl_analysis_boolean_values` |
| `text` | `tbl_analysis_notes` or `tbl_analysis_identifiers` |
| `int4range` | (year ranges — see below) |

There's also a purpose-built `tbl_analysis_dating_ranges` table (low/high integers plus
`age_type_id`, `dating_uncertainty_id`, `season_id`) that isn't tied to a single `base_type` —
it's used directly by dating-specific value classes, as the next few cells show.

Right now, **all 54 existing `tbl_value_classes` rows belong to Dendrochronology or Ancient DNA
methods** — none for radiocarbon at all. So using this chain for any of our five columns means
*creating* new value classes, not reusing existing ones.


In [12]:
value_classes = pd.read_sql(
    """
    select vc.value_class_id, vc.name, vt.name as value_type_name, vt.base_type, m.method_name
    from tbl_value_classes vc
    left join tbl_value_types vt on vt.value_type_id = vc.value_type_id
    left join tbl_methods m on m.method_id = vc.method_id
    order by vc.value_class_id
    """,
    engine,
)
print(f"{len(value_classes)} value classes defined; methods in use: {sorted(value_classes['method_name'].unique())}")
value_classes[['value_class_id', 'name', 'value_type_name', 'base_type']].head(10)


54 value classes defined; methods in use: ['Ancient DNA analysis (Sample Extraction/Analysis)', 'Dendrochronology']


,value_class_id,name,value_type_name,base_type
0,1,Tree species,Label,text
1,2,Tree rings,Count,integer
2,3,Earlywood/Latewood,Early/Latewood,category
3,4,Number of analysed radii.,Count,integer
4,5,EW/LW measurements,Boolean,boolean
5,6,Sapwood (Sp),Count,integer
6,7,Bark (B),Boolean,boolean
7,8,Waney edge (W),Waney edge (W),category
8,9,Pith (P),Count,integer
9,10,Tree age ≥,Age in years,integer


## pMC_value / pMC_error → new value classes with `base_type = decimal`

Both are decimal lab measurements (only ~100 of 30,301 Strucke rows even have them populated —
pMC is reported for a minority of samples). The closest existing precedent is the Ancient DNA
method's percentage/measurement value classes below, which already prove the "Percentage"/
"Measurement" value types round-trip correctly through `tbl_analysis_numerical_values`.

Proposed:
- **pMC_value** → new value class, `value_type` = *Percentage* (id 4, `decimal`) → `tbl_analysis_numerical_values.value`
- **pMC_error** → new value class, `value_type` = *Measurement* (id 11, `decimal`) → `tbl_analysis_numerical_values.value`
- Both tied to a raw-measurement method, e.g. `method_id` 38 (*C14 Accelerator dating*) or 39
  (*C14 Conventional*) — pMC is a pre-calibration lab output, not a calibration result.


In [13]:
decimal_precedent = pd.read_sql(
    """
    select vc.value_class_id, vc.name, m.method_name, vt.name as value_type_name, count(*) as n_rows
    from tbl_analysis_numerical_values anv
    join tbl_analysis_values av on av.analysis_value_id = anv.analysis_value_id
    join tbl_value_classes vc on vc.value_class_id = av.value_class_id
    join tbl_methods m on m.method_id = vc.method_id
    left join tbl_value_types vt on vt.value_type_id = vc.value_type_id
    group by vc.value_class_id, vc.name, m.method_name, vt.name
    order by n_rows desc
    """,
    engine,
)
decimal_precedent


,value_class_id,name,method_name,value_type_name,n_rows
0,48,Average depth of coverage - mtDNA (x),Ancient DNA analysis (Sample Extraction/Analysis),Measurement,10
1,50,Average depth of coverage - genome (x),Ancient DNA analysis (Sample Extraction/Analysis),Measurement,10
2,51,Average read length,Ancient DNA analysis (Sample Extraction/Analysis),Measurement,10
3,29,Duplicate reads (%),Ancient DNA analysis (Sample Extraction/Analysis),Percentage,5
4,26,3’ damage,Ancient DNA analysis (Sample Extraction/Analysis),Measurement,5
5,49,Breadth of coverage - genome (%),Ancient DNA analysis (Sample Extraction/Analysis),Percentage,5
6,30,Raw endogenous content (%),Ancient DNA analysis (Sample Extraction/Analysis),Percentage,5
7,27,5’ damage,Ancient DNA analysis (Sample Extraction/Analysis),Measurement,5
8,28,Short reads (%),Ancient DNA analysis (Sample Extraction/Analysis),Percentage,5


## cal_68_min/max, cal_95_min/max → `tbl_analysis_dating_ranges`, one value class per confidence level

These are exactly what `tbl_analysis_dating_ranges` was built for (`low_value`/`high_value`
integers, both non-null for 29,731 of 30,301 Strucke rows, always whole numbers — confirmed
separately against the raw CSV). `tbl_methods` already has explicit calibration methods (156
*Calibrated radiocarbon date (method unspecified)*, 157 *Calibrated AMS radiocarbon date*).

**Important finding: there is no existing "confidence level" reference table.** The
`dating_uncertainty_id` field (`tbl_dating_uncertainty`) only encodes qualifiers like "Ca.",
"<", ">", "From"/"To" — nothing about 68% vs. 95% statistical confidence. So cal_68 and cal_95
can't share one value class distinguished by `dating_uncertainty_id`; they need **two separate
value classes** (e.g. *"Calibrated date range (68% / 1σ)"* and *"Calibrated date range (95% /
2σ)"*), each producing its own `tbl_analysis_dating_ranges` row, both against the same
`analysis_entity_id`.

**Second finding: the existing `age_type_id` won't cleanly fit either.** Every one of the
7,775 existing `tbl_analysis_dating_ranges` rows (all Dendrochronology) uses `age_type_id = 1`
("AD"), and — consistent with that label — all their values are positive (890–1964). Strucke's
`cal_68`/`cal_95`/`median_cal_year` use astronomical year numbering with negative values for BC
dates (e.g. -930). Reusing "AD" as-is would misrepresent those; a new `tbl_age_types` row (e.g.
"cal BC/AD" or "cal BP") would be needed first.


In [14]:
dating_range_precedent = pd.read_sql(
    """
    select vc.value_class_id, vc.name, m.method_name, count(*) as n_rows
    from tbl_analysis_dating_ranges adr
    join tbl_analysis_values av on av.analysis_value_id = adr.analysis_value_id
    join tbl_value_classes vc on vc.value_class_id = av.value_class_id
    join tbl_methods m on m.method_id = vc.method_id
    group by vc.value_class_id, vc.name, m.method_name
    order by n_rows desc
    """,
    engine,
)
print('Existing tbl_analysis_dating_ranges usage (all Dendrochronology today):')
print(dating_range_precedent)

age_types = pd.read_sql('select * from tbl_age_types order by age_type_id', engine)
print()
print('tbl_age_types (only one row exists):')
print(age_types)

age_type_usage = pd.read_sql(
    """
    select adr.age_type_id, at.age_type, count(*) as n, min(adr.low_value) as min_val, max(adr.high_value) as max_val
    from tbl_analysis_dating_ranges adr
    left join tbl_age_types at on at.age_type_id = adr.age_type_id
    group by adr.age_type_id, at.age_type
    """,
    engine,
)
print()
print('age_type_id actually used, and the value range seen with it:')
print(age_type_usage)

dating_uncertainty = pd.read_sql('select dating_uncertainty_id, uncertainty, description from tbl_dating_uncertainty order by dating_uncertainty_id', engine)
print()
print('tbl_dating_uncertainty (qualifiers only, no confidence-level concept):')
print(dating_uncertainty)


Existing tbl_analysis_dating_ranges usage (all Dendrochronology today):
   value_class_id                             name       method_name  n_rows
0              17         Outermost tree-ring date  Dendrochronology    3663
1              14           Estimated felling year  Dendrochronology    3658
2              15  Possible estimated felling year  Dendrochronology     454

tbl_age_types (only one row exists):
   age_type_id age_type                                        description  \
0            1       AD  Anno Domini, Christian era; calendar era dates...   

                      date_updated  
0 2019-12-20 13:45:52.604010+00:00  

age_type_id actually used, and the value range seen with it:
   age_type_id age_type     n  min_val  max_val
0            1       AD  7775      890     1964

tbl_dating_uncertainty (qualifiers only, no confidence-level concept):
   dating_uncertainty_id uncertainty  \
0                      1         Ca.   
1                      2           <   
2

## median_cal_year → new value class with `value_type = Year` → `tbl_analysis_integer_values`

Always a whole number (confirmed against the raw CSV — 29,731 non-null, 0% with a fractional
part), so it's a plain point-estimate year rather than a range. The *Year* value type
(id 6, `base_type = integer`) already has a direct precedent in Dendrochronology's "Inferred
growth year ≥/≤" value classes below, which confirms `base_type = integer` value classes
route through `tbl_analysis_integer_values`, not `tbl_analysis_numerical_values`, in practice.

Same BC/AD sign caveat as above applies here too — whatever `age_type_id` ends up used for
cal_68/cal_95 should be reused for `median_cal_year` as well, for consistency.


In [15]:
integer_precedent = pd.read_sql(
    """
    select vc.value_class_id, vc.name, vt.name as value_type_name, vt.base_type,
      (select count(*) from tbl_analysis_integer_values iv join tbl_analysis_values av on av.analysis_value_id = iv.analysis_value_id where av.value_class_id = vc.value_class_id) as n_integer_values,
      (select count(*) from tbl_analysis_numerical_values nv join tbl_analysis_values av on av.analysis_value_id = nv.analysis_value_id where av.value_class_id = vc.value_class_id) as n_numerical_values
    from tbl_value_classes vc
    join tbl_value_types vt on vt.value_type_id = vc.value_type_id
    where vt.value_type_id in (5, 6)
    """,
    engine,
)
integer_precedent


,value_class_id,name,value_type_name,base_type,n_integer_values,n_numerical_values
0,10,Tree age ≥,Age in years,integer,2452,0
1,11,Tree age ≤,Age in years,integer,2469,0
2,12,Inferred growth year ≥,Year,integer,2294,0
3,13,Inferred growth year ≤,Year,integer,2294,0


## The full chain, and what's missing to actually populate it

`tbl_analysis_entities` is the join point between the dedicated `tbl_geochronology` row and any
generic `tbl_analysis_values` rows for the *same physical sample* — `c14_age_bp`/`c14_error`
would stay in `tbl_geochronology` exactly as today, while `pMC_value`/`pMC_error`/`cal_68`/
`cal_95`/`median_cal_year` would live in `tbl_analysis_values` rows sharing that same
`analysis_entity_id`.

To go from "structurally possible" to "actually populated," in order:

1. **Add 5 new `tbl_value_classes` rows** (one per target column) — see the table below.
2. **Add `tbl_dataset_methods` rows** wiring the dataset to whichever method(s) are actually
   used (a raw-measurement method for pMC, a calibration method for cal_68/95/median_cal_year)
   — currently **zero rows exist for any C14-related method**, confirmed earlier in this
   notebook, so this step is required regardless of which columns get mapped.
3. **Possibly add a new `tbl_age_types` row** (e.g. "cal BC/AD") if Strucke's negative-year
   convention shouldn't be forced into the existing "AD" type.
4. For each Strucke row: reuse the `analysis_entity_id` already established for that sample's
   `tbl_geochronology` row (or create one, if a sample has calibration data but no raw
   age/error), then insert one `tbl_analysis_values` row per non-null target column, each
   pointing at the matching new value class, with the actual value written into the
   appropriate subtype table.

| Strucke column | proposed `value_class` | `value_type` | subtype table | proposed `method` |
|---|---|---|---|---|
| `pMC_value` | *pMC value* (new) | Percentage (decimal) | `tbl_analysis_numerical_values` | C14 Accelerator dating / C14 Conventional |
| `pMC_error` | *pMC error* (new) | Measurement (decimal) | `tbl_analysis_numerical_values` | same as pMC_value |
| `cal_68_min`/`cal_68_max` | *Calibrated date range (68% / 1σ)* (new) | — (dating range) | `tbl_analysis_dating_ranges` (`low_value`/`high_value`) | Calibrated AMS radiocarbon date (157) |
| `cal_95_min`/`cal_95_max` | *Calibrated date range (95% / 2σ)* (new) | — (dating range) | `tbl_analysis_dating_ranges` (`low_value`/`high_value`) | same as cal_68 |
| `median_cal_year` | *Median calibrated year* (new) | Year (integer) | `tbl_analysis_integer_values` | same as cal_68 |

This is a design proposal, not a migration — nothing has been inserted into `sead_staging`.
Whoever owns the SEAD schema should confirm the exact method/age-type choices (particularly the
BC/AD sign handling) before any of this is actually written.
